# Threat-Aware Quantum-Secure Communication for Biomedical Networks

Runs the full pipeline top to bottom. All settings live in the cell below; no other cell hard-codes them. Simulation only: no real photons or hardware.

In [ ]:
# ---- SETTINGS: the only place these values are defined ----
# Bases are encoded as 0 = Z, 1 = X everywhere in this project.
SEED = 42                      # one fixed seed for Python, NumPy and the simulator (reproducible runs)
N_QUBITS = 2000                # qubits Alice sends per BB84 run (about half survive sifting)
QBER_SAMPLE_SIZE = 300         # sifted bits publicly compared to estimate QBER (these are then discarded)
SHOTS = 1                      # shots per circuit: each BB84 qubit is sent and measured once
NOISE_LEVELS = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20]   # depolarizing strength to sweep
ATTACK_FRACTIONS = [0.0, 0.25, 0.5, 0.75, 1.0]       # fraction f of qubits Eve intercepts
N_SEEDS = 20                   # repeats with different seeds, for error bars

THREAT_WINDOW_SIZE = 100       # test records averaged to get one channel threat level (one window of traffic)

# Phase 6 scenarios: share of real attack records used to build each 100-record window
WINDOW_ATTACK_SHARES = {"mostly normal": 0.10, "mixed": 0.50, "attack-heavy": 0.90}
CHANNELS = {"clean": (0.0, 0.0), "noisy": (0.10, 0.0),        # (noise level, attack fraction f)
            "weak Eve": (0.0, 0.3), "full Eve": (0.0, 1.0)}

# E91 extension
E91_ROUNDS = 2000             # E91 rounds per run (like N_QUBITS for BB84)
# INJECTED, ARTIFICIAL faults (not real attacks): extra depolarizing noise applied to ONE protocol only
INJECTED_BB84_FAULT_NOISE = 0.40   # BB84-only fault: BB84 sees this noise, E91 stays honest
INJECTED_E91_FAULT_NOISE = 0.50    # E91-only fault: E91 sees this noise, BB84 stays honest
MORE_SAMPLES_FACTOR = 4            # 'Monitor' = rerun with more samples: multiply the sample sizes by this

TRAIN_PATH = "data/KDDTrain+.txt"   # NSL-KDD training file (no header row)
TEST_PATH = "data/KDDTest+.txt"     # NSL-KDD test file (no header row)

## Check the environment (task 0.2)

In [ ]:
import sys, qiskit, qiskit_aer, sklearn, pandas, matplotlib, numpy
print("Python      ", sys.version.split()[0])
for m in (qiskit, qiskit_aer, sklearn, pandas, matplotlib, numpy):
    print(f"{m.__name__:12}", m.__version__)

## Phase 1: Threat classifier

### 1.1 Load and inspect

In [ ]:
from src.classifier import *
from src.decision import threat_level
train_df = load_nsl_kdd(TRAIN_PATH)
test_df = load_nsl_kdd(TEST_PATH)
print("train rows, columns:", train_df.shape)
print("test rows, columns: ", test_df.shape)
for name, df in (("train", train_df), ("test", test_df)):
    y = make_binary_label(df)
    print(f"{name}: normal = {(y == 0).sum()}, attack = {(y == 1).sum()}, distinct labels = {df['label'].nunique()}")
unseen = sorted(set(test_df["label"]) - set(train_df["label"]))
print("attack types only in test:", len(unseen), unseen)

### 1.2 Preprocess (one-hot, scaler fitted on train only)

In [ ]:
X_train, X_test, y_train, y_test = preprocess(train_df, test_df)
print("X_train", X_train.shape, "| X_test", X_test.shape)
assert list(X_train.columns) == list(X_test.columns)
assert X_train.shape[1] == X_test.shape[1]
assert not X_train.isna().any().any() and not X_test.isna().any().any()
print("columns match, no missing values")

### 1.3 and 1.4 Train both models, test-set metrics

In [ ]:
import pandas as pd
logreg = train_logistic_regression(X_train, y_train, SEED)
forest = train_random_forest(X_train, y_train, SEED)
results = {"Logistic regression": evaluate(logreg, X_test, y_test),
           "Random forest": evaluate(forest, X_test, y_test)}
metrics = pd.DataFrame({n: {k: v for k, v in r.items() if k != "confusion_matrix"} for n, r in results.items()}).T
metrics.index.name = "model"
print(metrics.round(4))
for n, r in results.items():
    print(n, "confusion matrix [[TN FP] [FN TP]]:", r["confusion_matrix"].tolist())
metrics.to_csv("results/metrics.csv")

### 1.5 Threat probability and threat levels

A channel's threat is the **average** random-forest probability over a window of `THREAT_WINDOW_SIZE` consecutive test records.

In [ ]:
assert threat_level(0.1) == "Low" and threat_level(0.5) == "Medium" and threat_level(0.9) == "High"
print("threat_level checks passed")
p_test = threat_probability(forest, X_test)
print("single records:", pd.Series([threat_level(p) for p in p_test]).value_counts().to_dict())
p_window = window_threat(p_test, THREAT_WINDOW_SIZE)
print(f"windows of {THREAT_WINDOW_SIZE} records: {len(p_window)} windows,",
      pd.Series([threat_level(p) for p in p_window]).value_counts().to_dict())

### 1.6 Confusion matrix plot

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, r) in zip(axes, results.items()):
    ax.imshow(r["confusion_matrix"], cmap="Blues")
    for (i, j), v in __import__("numpy").ndenumerate(r["confusion_matrix"]):
        ax.text(j, i, v, ha="center", va="center")
    ax.set_xticks([0, 1], ["normal", "attack"]); ax.set_yticks([0, 1], ["normal", "attack"])
    ax.set_xlabel("Predicted label (test records)"); ax.set_ylabel("True label (test records)")
    ax.set_title(f"{name}: test set")
fig.tight_layout()
fig.savefig("results/confusion_matrix.png", dpi=150)
plt.show()

## Phase 2: BB84 core (ideal channel)

### 2.1 Alice: random bits and bases (0 = Z, 1 = X)

In [ ]:
import numpy as np
from src.bb84 import alice_random, build_circuit, run_circuits, sift, estimate_qber, run_bb84
a1 = alice_random(N_QUBITS, np.random.default_rng(SEED))
a2 = alice_random(N_QUBITS, np.random.default_rng(SEED))
assert (a1[0] == a2[0]).all() and (a1[1] == a2[1]).all()
print("same seed gives same bits and bases: OK")
print("first 10 bits :", a1[0][:10], "\nfirst 10 bases:", a1[1][:10])

### 2.2 Encode qubits: four example circuits (the `id` is the channel step)

In [ ]:
for bit in (0, 1):
    for basis, name in ((0, "Z"), (1, "X")):
        print(f"bit {bit}, Alice basis {name}  (Bob measures in Z here)")
        print(build_circuit(bit, basis, 0).draw("text"))

### 2.3 Bob measures: one batched run, one result bit per qubit

In [ ]:
rng = np.random.default_rng(SEED)
alice_bits, alice_bases = alice_random(N_QUBITS, rng)
bob_bases = rng.integers(0, 2, size=N_QUBITS)  # 0 = Z, 1 = X
circuits = [build_circuit(b, a, c) for b, a, c in zip(alice_bits, alice_bases, bob_bases)]
bob_bits = run_circuits(circuits, SHOTS, SEED)
assert len(bob_bits) == N_QUBITS
print("qubits sent:", N_QUBITS, "| Bob result bits:", len(bob_bits))

### 2.4 Sifting

In [ ]:
alice_sifted, bob_sifted = sift(alice_bits, bob_bits, alice_bases, bob_bases)
print(f"sifted: {len(alice_sifted)} of {N_QUBITS} ({len(alice_sifted) / N_QUBITS:.1%} kept)")

### 2.5 QBER estimation

In [ ]:
res = estimate_qber(alice_sifted, bob_sifted, QBER_SAMPLE_SIZE, rng)
print(res)

### 2.6 Validation: ideal channel (no noise, no eavesdropper) must give QBER = 0 exactly

In [ ]:
for s in range(SEED, SEED + N_SEEDS):
    r = run_bb84(N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, s)
    print(f"seed {s}: QBER = {r.qber}, mismatches = {r.mismatches}/{r.sample_size}, sifted = {r.sifted_count}, raw key = {r.raw_key_length}")
    assert r.qber == 0, "BUG: ideal channel must give QBER = 0"
assert run_bb84(N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, SEED) == run_bb84(N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, SEED)
print("ideal-channel check passed; same seed reproduces the same result")

## Phase 3: Channel noise

### 3.1 Depolarizing noise on the channel step

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from src.noise import build_noise_model
from src.bb84 import qber_standard_error, check_label

# transpile must keep the channel gate (id), otherwise noise has nothing to act on
kept = transpile(build_circuit(0, 0, 0), AerSimulator(), optimization_level=0).count_ops()
assert "id" in kept
print("channel gate survives transpile:", dict(kept))

# Quick test: every encoding (bit x Alice basis), Bob measuring in the SAME basis.
# Ideal channel -> 0 errors. Noisy channel -> errors must appear for ALL four encodings.
TEST_LEVEL = max(NOISE_LEVELS)
noisy = build_noise_model(TEST_LEVEL)
print(f"error rate per encoding at noise level {TEST_LEVEL} ({N_QUBITS} qubits each):")
for bit in (0, 1):
    for basis, name in ((0, "Z"), (1, "X")):
        circs = [build_circuit(bit, basis, basis)] * N_QUBITS
        ideal_err = (run_circuits(circs, SHOTS, SEED) != bit).mean()
        noisy_err = (run_circuits(circs, SHOTS, SEED, noisy) != bit).mean()
        print(f"  bit {bit}, basis {name}: ideal = {ideal_err:.4f}, noisy = {noisy_err:.4f}")
        assert ideal_err == 0 and noisy_err > 0

### 3.2 Noise sweep (no eavesdropper)

QBER is pooled over `N_SEEDS` seeds. The "theory" column is noise level / 2 (explained in the README), shown only for comparison; the QBER columns are what the simulation produced.

In [ ]:
import pandas as pd
rows = []
noise_qbers = {}   # per-seed QBERs for each noise level (used for error bars)
for level in NOISE_LEVELS:
    model = build_noise_model(level)
    runs = [run_bb84(N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, s, model) for s in range(SEED, SEED + N_SEEDS)]
    noise_qbers[level] = [r.qber for r in runs]
    pooled = sum(r.mismatches for r in runs) / sum(r.sample_size for r in runs)
    expected = level / 2   # theory: depolarizing noise flips a measured bit with probability level / 2
    rows.append({"noise_level": level,
                 f"QBER seed {SEED} (%)": 100 * runs[0].qber,
                 f"QBER all {N_SEEDS} seeds (%)": 100 * pooled,
                 "theory (%)": 100 * expected,
                 "check (2 SE)": check_label(pooled, expected, N_SEEDS * QBER_SAMPLE_SIZE)})
sweep = pd.DataFrame(rows)
print(sweep.round(2).to_string(index=False))
means = sweep[f"QBER all {N_SEEDS} seeds (%)"]
print("QBER increases with noise strength:", "PASS" if (means.diff().dropna() > 0).all() else "MISS")

## Phase 4: Eavesdropper (intercept-resend)

### 4.1 Eve's circuit and how we read the two classical bits

In [ ]:
from src.bb84 import eve_choices
# Same qubit (bit 0, Alice Z, Bob Z), Eve attacks in Z, then in X:
for eb, name in ((0, "Z"), (1, "X")):
    print(f"Eve attacks in basis {name} (clbit 1 = Eve, clbit 0 = Bob):")
    print(build_circuit(0, 0, 0, eve_basis=eb).draw("text"))

# Reading the bits: key = "<clbit1><clbit0>" = "<Eve><Bob>", Bob is the RIGHTMOST character.
# Check: when Eve's basis equals Alice's she must read Alice's bit exactly.
rng = np.random.default_rng(SEED)
bits, a_bases = alice_random(N_QUBITS, rng)
e_bases = rng.integers(0, 2, size=N_QUBITS)
circs = [build_circuit(b, a, a, e) for b, a, e in zip(bits, a_bases, e_bases)]
bob_b, eve_b = run_circuits(circs, SHOTS, SEED, return_eve=True)
same = a_bases == e_bases
print(f"Eve right-basis qubits: {same.sum()}, Eve's bit equals Alice's on all of them: {(eve_b[same] == bits[same]).all()}")
print(f"Eve wrong-basis qubits: {(~same).sum()}, Eve's bit equals Alice's on {(eve_b[~same] == bits[~same]).mean():.1%} (should be about half)")
print(f"Bob (same basis as Alice) wrong on {(bob_b != bits).mean():.1%} of ALL qubits when Eve attacks all of them")
assert (eve_b[same] == bits[same]).all()

### 4.2 Validation: full attack (f = 1, no noise)

Tolerance: within 2 standard errors, SE = sqrt(p(1-p)/n), p = expected QBER, n = total sampled bits (all seeds pooled).

In [ ]:
def pooled_qber(f, noise_model=None):
    runs = [run_bb84(N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, s, noise_model, eve_fraction=f)
            for s in range(SEED, SEED + N_SEEDS)]
    return runs, sum(r.mismatches for r in runs) / sum(r.sample_size for r in runs)

n_total = N_SEEDS * QBER_SAMPLE_SIZE
runs, q = pooled_qber(1.0)
se = qber_standard_error(0.25, n_total)
print(f"f = 1: per-seed QBER (%) = {[round(100 * r.qber, 1) for r in runs]}")
print(f"f = 1: pooled QBER = {100 * q:.2f}%  (theory 25%, 2 SE = {200 * se:.2f} points, n = {n_total})")
print("full-attack check:", check_label(q, 0.25, n_total))

### 4.3 Validation: partial attack sweep (no noise)

The "theory" column is 0.25 x f, for comparison only.

In [ ]:
rows = []
eve_qbers = {}   # per-seed QBERs for each attack fraction (used for error bars)
for f in ATTACK_FRACTIONS:
    runs, q = pooled_qber(f)
    eve_qbers[f] = [r.qber for r in runs]
    rows.append({"f": f, f"QBER seed {SEED} (%)": 100 * runs[0].qber, f"QBER all {N_SEEDS} seeds (%)": 100 * q,
                 "theory 0.25*f (%)": 100 * 0.25 * f,
                 "check (2 SE)": check_label(q, 0.25 * f, n_total)})
eve_sweep = pd.DataFrame(rows)
print(eve_sweep.round(2).to_string(index=False))
print("partial-attack sweep: see the check column above (a MISS is reported, not hidden)")

## Phase 5: Decision rules

### 5.1 Threshold table

In [ ]:
from src.decision import BB84_THRESHOLDS, decide_bb84
print("5.1 threshold table (QBER fractions):")
print(pd.DataFrame(BB84_THRESHOLDS).T)

### 5.2 `decide_bb84` grid test and edge cases

In [ ]:
# 5.2 the 9-cell grid from CLAUDE.md section 9.2
GRID = {0.04: ("Accept", "Accept", "Monitor"),
        0.06: ("Accept", "Monitor", "Monitor"),
        0.09: ("Monitor", "Monitor", "Reject")}
for q, expected in GRID.items():
    got = tuple(decide_bb84(q, lvl) for lvl in ("Low", "Medium", "High"))
    print(f"QBER {q}: {got}")
    assert got == expected
print("9-cell grid: all match")

# Edge cases exactly at the limits (strict < and > means a limit value is Monitor)
print("\nEdge cases:")
for lvl, lim in BB84_THRESHOLDS.items():
    for name in ("accept", "reject"):
        print(f"  {lvl:6} QBER exactly {lim[name]:.2f} ({name} limit): {decide_bb84(lim[name], lvl)}")

## Phase 6: Integration

### 6.1 One record, end to end

In [ ]:
from src.decision import run_scenario
p_test = threat_probability(forest, X_test)   # random forest threat probability per test record

noise, f = CHANNELS["noisy"]   # same noisy channel for both records, so only the threat level differs
def show(title, p):
    r = run_scenario(p, noise, f, N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, SEED)
    print(f"{title}: threat probability {p:.3f} -> level {r['threat_level']} -> BB84 (noise {noise}, f = {f}) QBER {100 * r['qber']:.1f}% -> {r['decision']}")

y_arr = y_test.to_numpy()
normal_row = int(np.where((y_arr == 0) & (p_test < 0.33))[0][0])   # first real normal record with low probability
attack_row = int(np.where((y_arr == 1) & (p_test > 0.66))[0][0])   # first real attack record with high probability
show(f"real normal record (row {normal_row})", p_test[normal_row])
show(f"real attack record (row {attack_row})", p_test[attack_row])

### 6.2 Scenario table: 3 chosen windows x 4 channels

Each window is built from chosen test traffic; its threat level is the average random-forest probability. Each scenario is one BB84 run at `SEED`.

In [ ]:
rng_w = np.random.default_rng(SEED)
windows = {}
for name, share in WINDOW_ATTACK_SHARES.items():
    p_win = build_window(p_test, y_test, share, THREAT_WINDOW_SIZE, rng_w)
    windows[name] = p_win.mean()
    print(f"window '{name}': {share:.0%} real attacks -> average probability {p_win.mean():.3f} -> {threat_level(p_win.mean())}")

rows = []
for wname, p_avg in windows.items():
    for cname, (noise, f) in CHANNELS.items():
        r = run_scenario(p_avg, noise, f, N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, SEED)
        rows.append({"scenario": f"{wname} + {cname}", "window": wname, "avg_threat_prob": round(p_avg, 4),
                     "noise_level": noise, "f": f, "threat_level": r["threat_level"],
                     "qber": r["qber"], "decision": r["decision"]})
scenarios = pd.DataFrame(rows)
scenarios.to_csv("results/scenarios.csv", index=False)
show_tbl = scenarios.assign(qber_pct=(100 * scenarios["qber"]).round(2)).drop(columns=["qber", "window"])
print(show_tbl.to_string(index=False))

# What-if table: only needed if no real window reached High. Hypothetical, bins unchanged.
if "High" not in set(scenarios["threat_level"]):
    print("\nNO window reached High. WHAT-IF (HYPOTHETICAL, not produced by any real window): same QBERs judged at High")
    whatif = scenarios[["scenario", "noise_level", "f", "threat_level", "qber", "decision"]].copy()
    whatif["decision_if_High"] = [decide_bb84(q, "High") for q in whatif["qber"]]
    whatif["qber_pct"] = (100 * whatif["qber"]).round(2)
    print(whatif.drop(columns=["qber"]).to_string(index=False))
    whatif.to_csv("results/scenarios_whatif_high.csv", index=False)

### Weak Eve over 20 seeds

The weak-Eve scenario above is one seed-42 run. This is the average over all seeds.

In [ ]:
noise_w, f_w = CHANNELS["weak Eve"]
weak = [run_bb84(N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, s, build_noise_model(noise_w), f_w) for s in range(SEED, SEED + N_SEEDS)]
weak_q = np.array([r.qber for r in weak])
print(f"f = {f_w}, {N_SEEDS} seeds: mean QBER = {100 * weak_q.mean():.2f}%, std = {100 * weak_q.std(ddof=1):.2f} points, theory 0.25*f = {100 * 0.25 * f_w:.2f}%")
print("seed-42 run alone:", f"{100 * weak_q[0]:.2f}%")
print("check:", check_label(weak_q.mean(), 0.25 * f_w, N_SEEDS * QBER_SAMPLE_SIZE))
print("decision for the 20-seed mean QBER ->", {lvl: decide_bb84(weak_q.mean(), lvl) for lvl in ("Low", "Medium", "High")})
print("share of the 20 single runs by decision at Low:", pd.Series([decide_bb84(q, "Low") for q in weak_q]).value_counts().to_dict())

## Phase 7: Plots

Error bars are the sample standard deviation of the per-seed QBER across the 20 seeds. Colours: Accept green, Monitor amber, Reject red.

In [ ]:
import matplotlib.pyplot as plt
GREEN, AMBER, RED = "#2e9e5b", "#f0a202", "#d64545"
DECISION_COLOUR = {"Accept": GREEN, "Monitor": AMBER, "Reject": RED}

def sweep_plot(x_values, qbers_by_x, theory_fn, theory_label, xlabel, title, path):
    """Measured mean QBER (%) with std error bars over seeds, plus a labelled theory line."""
    xs = np.array(x_values)
    means = np.array([np.mean(qbers_by_x[x]) for x in xs]) * 100
    stds = np.array([np.std(qbers_by_x[x], ddof=1) for x in xs]) * 100
    fig, ax = plt.subplots(figsize=(6, 4.2))
    grid = np.linspace(xs.min(), xs.max(), 100)
    ax.plot(grid, 100 * theory_fn(grid), "--", color="gray", label=theory_label)
    ax.errorbar(xs, means, yerr=stds, fmt="o", color="#1f5fa8", capsize=4, label=f"measured (mean ± std, {N_SEEDS} seeds)")
    ax.set_xlabel(xlabel); ax.set_ylabel("QBER (%)"); ax.set_title(title)
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.show()
    return means, stds

# 7.1 and 7.5 (error bars)
m1, s1 = sweep_plot(ATTACK_FRACTIONS, eve_qbers, lambda x: 0.25 * x, "theory (0.25 × f)",
                    "Eve attack fraction f (share of qubits intercepted)", "QBER vs eavesdropper attack fraction (no noise)",
                    "results/qber_vs_eve_fraction.png")
print("7.1 mean QBER (%):", dict(zip(ATTACK_FRACTIONS, m1.round(2))), "| std (points):", s1.round(2))

In [ ]:
# 7.2 (error bars included)
m2, s2 = sweep_plot(NOISE_LEVELS, noise_qbers, lambda x: x / 2, "theory (noise level / 2)",
                    "Depolarizing noise level (error probability on the channel)", "QBER vs channel noise (no eavesdropper)",
                    "results/qber_vs_noise.png")
print("7.2 mean QBER (%):", dict(zip(NOISE_LEVELS, m2.round(2))), "| std (points):", s2.round(2))

In [ ]:
# 7.3 decision map: for each threat level, which decision every QBER value would get
q_grid = np.linspace(0, 0.30, 3001)
levels = ["High", "Medium", "Low"]   # bottom to top
fig, ax = plt.subplots(figsize=(8, 3.6))
for row, lvl in enumerate(levels):
    decisions = np.array([decide_bb84(q, lvl) for q in q_grid])
    for d in ("Accept", "Monitor", "Reject"):
        mask = decisions == d
        ax.barh(row, 100 * (q_grid[1] - q_grid[0]) * mask.sum(), left=100 * q_grid[mask][0], height=0.7, color=DECISION_COLOUR[d], edgecolor="white")
        ax.text(100 * q_grid[mask].mean(), row, d, ha="center", va="center", fontsize=9, color="black")
ax.set_yticks(range(3), levels); ax.set_xlabel("QBER (%)"); ax.set_ylabel("Threat level")
ax.set_title("Decision map: threat level vs QBER"); ax.set_xlim(0, 100 * q_grid.max())
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in DECISION_COLOUR.values()], labels=list(DECISION_COLOUR), loc="upper right", ncol=3, bbox_to_anchor=(1, -0.18))
fig.tight_layout(); fig.savefig("results/decision_map.png", dpi=150); plt.show()

In [ ]:
# 7.4 theory curve: usable key fraction 1 - 2h(Q)
def binary_entropy(q):
    q = np.clip(q, 1e-12, 1 - 1e-12)
    return -q * np.log2(q) - (1 - q) * np.log2(1 - q)
q = np.linspace(0, 0.20, 2001)
key_fraction = np.maximum(0, 1 - 2 * binary_entropy(q))
zero_q = q[np.argmax(key_fraction == 0)]   # first Q where the theory curve is zero
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(100 * q, key_fraction, color="#1f5fa8", label="theory: 1 − 2h(Q)")
ax.axvline(100 * zero_q, ls="--", color=RED, label=f"theory reaches zero at Q ≈ {100 * zero_q:.1f}%")
ax.set_xlabel("QBER Q (%)"); ax.set_ylabel("Usable key fraction (theory, asymptotic)")
ax.set_title("Theoretical usable key fraction vs QBER"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("results/key_rate_curve.png", dpi=150); plt.show()
print(f"7.4 theory curve first reaches zero at Q = {100 * zero_q:.2f}%")

## E91 extension (branch `e91`)

Simplified "E91-style" protocol on the same channel as BB84. E91 is equivalent to BB84 for key security, so this is an independent diagnostic, not a stronger key. Both protocols share the same simulated channel, so their evidence is correlated.

### Step 2a: the four CHSH settings alone, ideal channel

Each correlator E(a, b) is the average of (Alice's result x Bob's result), each result being +1 or -1. The theory column is cos(a - b).

In [ ]:
from src.dual_protocol_outline import (Channel, e91_run, ideal_correlator, s_check_label,
                                       ALICE_ANGLES, BOB_ANGLES, CHSH_PAIRS)
r = e91_run(Channel(0.0, 0.0, SEED), E91_ROUNDS)
print(f"ideal channel, seed {SEED}, {E91_ROUNDS} rounds")
for i, (name, (e, se, n)) in enumerate(r.correlators.items()):
    a, b, _ = CHSH_PAIRS[i]
    theory = ideal_correlator(ALICE_ANGLES[a], BOB_ANGLES[b])
    print(f"  {name}: measured {e:+.3f} +/- {se:.3f} (n = {n})   theory {theory:+.3f}   {s_check_label(e, theory, se)}")
S_MAX = 2 * np.sqrt(2)   # quantum maximum, 2*sqrt(2) = 2.83
print(f"S = {r.s:.3f} +/- {r.s_se:.3f}   theory {S_MAX:.3f}   {s_check_label(r.s, S_MAX, r.s_se)}")
print(f"matched key rounds: {r.key_rounds}, mismatches: {r.key_mismatches}")
assert r.key_mismatches == 0, "BUG: ideal channel key rounds must agree exactly"

### Step 2b: S vs attack fraction f over 20 seeds (no noise)

Per seed we get S and its standard error. Over the seeds, the mean S has standard error sqrt(sum of the per-seed variances) / number of seeds. The theory column is 2.83 x (1 - f/2). f = 0 is the ideal channel.

In [ ]:
import pandas as pd
def pooled_s(noise, f):
    """Mean S over N_SEEDS seeds, its standard error, and the spread (std) across seeds."""
    runs = [e91_run(Channel(noise, f, s), E91_ROUNDS) for s in range(SEED, SEED + N_SEEDS)]
    s_vals = np.array([x.s for x in runs])
    se_mean = float(np.sqrt(sum(x.s_se ** 2 for x in runs)) / N_SEEDS)
    return s_vals, se_mean

e91_s = {}   # f -> per-seed S values (kept for plots)
rows = []
for f in ATTACK_FRACTIONS:
    s_vals, se_mean = pooled_s(0.0, f)
    e91_s[f] = s_vals
    theory = S_MAX * (1 - f / 2)
    rows.append({"f": f, "S seed 42": s_vals[0], f"mean S ({N_SEEDS} seeds)": s_vals.mean(), "SE of mean": se_mean,
                 "std across seeds": s_vals.std(ddof=1), "theory 2.83*(1-f/2)": theory,
                 "check (2 SE)": s_check_label(s_vals.mean(), theory, se_mean)})
e91_table = pd.DataFrame(rows)
print(e91_table.round(3).to_string(index=False))
print("S falls with f:", "PASS" if (e91_table[f"mean S ({N_SEEDS} seeds)"].diff().dropna() < 0).all() else "MISS")

### Step 2c: the same noise model on E91 (depolarizing noise, no Eve)

Theory (derived, for comparison only): with probability = noise level the travelling qubit is replaced by a random one, which scales every correlator by (1 - noise level), so S is about 2.83 x (1 - noise level).

In [ ]:
rows = []
for noise in NOISE_LEVELS:
    s_vals, se_mean = pooled_s(noise, 0.0)
    theory = S_MAX * (1 - noise)
    rows.append({"noise level": noise, f"mean S ({N_SEEDS} seeds)": s_vals.mean(), "SE of mean": se_mean,
                 "theory 2.83*(1-noise)": theory, "check (2 SE)": s_check_label(s_vals.mean(), theory, se_mean)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

### Step 3a: unit tests (CI verdicts, E91 verdict, cross-check matrix)

In [ ]:
import tests.test_verdicts as t1, tests.test_cross_check as t2
for module in (t1, t2):
    for name in sorted(n for n in dir(module) if n.startswith("test_")):
        getattr(module, name)()
print("all unit tests passed (every matrix row at all three threat levels, both ANOMALY rows)")

### Step 3b: real end-to-end cases (BB84 + E91 on the same channel, seed 42)

Each protocol gives a verdict (Pass / Marginal / Fail); the cross-check matrix combines them with the threat level. Monitor means: rerun with more samples.

In [ ]:
from src.dual_protocol_outline import decide
rows = []
for cname, (noise, f) in CHANNELS.items():
    for level in ("Low", "Medium", "High"):
        d = decide(Channel(noise, f, SEED), level, N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, E91_ROUNDS)
        rows.append({"channel": cname, "noise": noise, "f": f, "threat_level": level,
                     "qber": d.bb84.qber, "mismatches": d.bb84.mismatches, "sample_size": d.bb84.sample_size,
                     "bb84_verdict": d.bb84_verdict, "S": d.e91.s, "S_se": d.e91.s_se, "e91_verdict": d.e91_verdict,
                     "decision": d.decision, "anomaly": d.anomaly})
real_cases = pd.DataFrame(rows)
real_cases.to_csv("results/crosscheck_scenarios.csv", index=False)
show = real_cases.assign(QBER=lambda x: (100 * x["qber"]).round(2).astype(str) + "%", S=lambda x: x["S"].round(3).astype(str) + " +/- " + x["S_se"].round(3).astype(str))
print(show[["channel", "threat_level", "QBER", "bb84_verdict", "S", "e91_verdict", "decision", "anomaly"]].to_string(index=False))

### Step 3c: INJECTED, ARTIFICIAL faults to demonstrate the two ANOMALY rows

**These are not real attacks.** Each fault adds extra noise to one protocol only, on top of an honest channel, so the other protocol still sees the honest channel:
- **Injected BB84-only fault** (e.g. a bad detector in the BB84 apparatus): BB84's QBER rises, E91 is unaffected. Expected: BB84 Fail + E91 Pass.
- **Injected E91-only fault** (e.g. a faulty entangled-pair source): E91's S drops, BB84 is unaffected. Expected: BB84 Pass + E91 Fail.

In a real system both protocols would see the same channel; this is only a demonstration of the matrix.

In [ ]:
from src.dual_protocol_outline import injected_bb84_only_fault, injected_e91_only_fault
honest = Channel(0.0, 0.0, SEED)
faults = {"INJECTED BB84-only fault": injected_bb84_only_fault(honest, INJECTED_BB84_FAULT_NOISE),
          "INJECTED E91-only fault": injected_e91_only_fault(honest, INJECTED_E91_FAULT_NOISE)}
rows = []
for fname, (bb84_ch, e91_ch) in faults.items():
    for level in ("Low", "Medium", "High"):
        d = decide(bb84_ch, level, N_QUBITS, QBER_SAMPLE_SIZE, SHOTS, E91_ROUNDS, e91_channel=e91_ch)
        rows.append({"injected_fault": fname, "threat_level": level, "qber": d.bb84.qber, "mismatches": d.bb84.mismatches,
                     "sample_size": d.bb84.sample_size, "bb84_verdict": d.bb84_verdict, "S": d.e91.s, "S_se": d.e91.s_se,
                     "e91_verdict": d.e91_verdict, "decision": d.decision, "anomaly": d.anomaly})
injected = pd.DataFrame(rows)
injected.to_csv("results/crosscheck_injected_faults.csv", index=False)
show = injected.assign(QBER=lambda x: (100 * x["qber"]).round(2).astype(str) + "%", S=lambda x: x["S"].round(3).astype(str) + " +/- " + x["S_se"].round(3).astype(str))
print("ALL ROWS BELOW ARE INJECTED, ARTIFICIAL FAULTS, NOT REAL ATTACKS")
print(show[["injected_fault", "threat_level", "QBER", "bb84_verdict", "S", "e91_verdict", "decision", "anomaly"]].to_string(index=False))

### Step 3d: what Monitor means: rerun with more samples

Same cases that were not decisive at High, rerun with the sample sizes multiplied by `MORE_SAMPLES_FACTOR` (a fresh run, so numbers differ from above).

In [ ]:
k = MORE_SAMPLES_FACTOR
d = decide(honest, "High", N_QUBITS * k, QBER_SAMPLE_SIZE * k, SHOTS, E91_ROUNDS * k)
print(f"clean channel, High, {k}x samples: QBER {100 * d.bb84.qber:.2f}% -> {d.bb84_verdict} | S {d.e91.s:.3f} +/- {d.e91.s_se:.3f} -> {d.e91_verdict} | decision {d.decision}, anomaly {d.anomaly}")
b, e = faults["INJECTED BB84-only fault"]
d = decide(b, "High", N_QUBITS * k, QBER_SAMPLE_SIZE * k, SHOTS, E91_ROUNDS * k, e91_channel=e)
print(f"INJECTED BB84-only fault, High, {k}x samples: QBER {100 * d.bb84.qber:.2f}% -> {d.bb84_verdict} | S {d.e91.s:.3f} +/- {d.e91.s_se:.3f} -> {d.e91_verdict} | decision {d.decision}, anomaly {d.anomaly}")

### Step 4a: plot of S vs attack fraction f (20 seeds)

In [ ]:
fs = np.array(ATTACK_FRACTIONS)
s_means = np.array([e91_s[f].mean() for f in fs])
s_stds = np.array([e91_s[f].std(ddof=1) for f in fs])
fig, ax = plt.subplots(figsize=(6, 4.2))
grid = np.linspace(fs.min(), fs.max(), 100)
ax.plot(grid, S_MAX * (1 - grid / 2), "--", color="gray", label="theory: 2.83 × (1 − f/2)")
ax.axhline(2.0, ls="--", color=RED, label="classical bound S = 2")
ax.errorbar(fs, s_means, yerr=s_stds, fmt="o", color="#1f5fa8", capsize=4, label=f"measured (mean ± std, {N_SEEDS} seeds)")
ax.set_xlabel("Eve attack fraction f (share of rounds intercepted)"); ax.set_ylabel("CHSH score S")
ax.set_title("E91: Bell score S vs eavesdropper attack fraction (no noise)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("results/s_vs_eve_fraction.png", dpi=150); plt.show()
print("mean S:", dict(zip(ATTACK_FRACTIONS, s_means.round(3))), "| std across seeds:", s_stds.round(3))

### Step 4b: dual-protocol decision vs BB84-only decision (same 12 real cases as step 3b)

BB84-only uses `decide_bb84` on the QBER point estimate (the original rule). Dual-protocol uses the 95% interval verdicts of both protocols and the cross-check matrix. Strictness order: Accept < Monitor < Reject.

In [ ]:
from src.decision import decide_bb84
order = {"Accept": 0, "Monitor": 1, "Reject": 2}
cmp_tbl = real_cases[["channel", "threat_level", "decision"]].rename(columns={"decision": "dual_protocol"}).copy()
cmp_tbl.insert(2, "bb84_only", [decide_bb84(q, l) for q, l in zip(real_cases["qber"], real_cases["threat_level"])])
diff = [order[d] - order[b] for b, d in zip(cmp_tbl["bb84_only"], cmp_tbl["dual_protocol"])]
cmp_tbl["dual vs bb84_only"] = ["stricter" if x > 0 else "more lenient" if x < 0 else "same" for x in diff]
cmp_tbl.to_csv("results/crosscheck_vs_bb84_only.csv", index=False)
print(cmp_tbl.to_string(index=False))
print("stricter:", diff.count(1) + diff.count(2), "| same:", diff.count(0), "| more lenient:", sum(x < 0 for x in diff))